# Inference

## Imports

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageDraw

from model.Model import MiniFCOSFaceV1
from dataset.Dataset import WiderFaceDataset, detection_collate
from preprocess import preprocess_own_image
from postprocess import decode_predictions, nms, box_iou_one_to_many, boxes_to_original_image

## Loading best model

In [ ]:
best_checkpoint_path = "../train/checkpoints/minifcos_face_v1_best.pt"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint = torch.load(
    best_checkpoint_path,
    map_location=device
)

model = MiniFCOSFaceV1().to(device)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

In [ ]:
import matplotlib.pyplot as plt

IMAGE_PATH = ''
IMAGE_SIZE = 320
SCORE_THRESHOLD = 0.15
NMS_IOU_THRESHOLD = 0.4


original_image, image_tensor, transform_info = preprocess_own_image(
    IMAGE_PATH,
    image_size=IMAGE_SIZE
)

prediction = model(image_tensor.unsqueeze(0).to(device))[0]

predicted_boxes_320, predicted_scores = decode_predictions(
    prediction = prediction
    image_tensor=image_tensor,
    device=device,
    image_size=IMAGE_SIZE,
    feature_size=40,
    score_threshold=SCORE_THRESHOLD,
    nms_iou_threshold=NMS_IOU_THRESHOLD
)

predicted_boxes_original = boxes_to_original_image(
    predicted_boxes_320,
    transform_info
)

print("Number of predictions:", len(predicted_boxes_original))

result_image = original_image.copy()
draw = ImageDraw.Draw(result_image)

for box, score in zip(predicted_boxes_original, predicted_scores):
    x1, y1, x2, y2 = box.tolist()

    draw.rectangle(
        [x1, y1, x2, y2],
        outline="black",
        width=3
    )

    draw.text(
        (x1, max(0, y1 - 15)),
        f"face {score.item():.3f}",
        fill="black"
    )

plt.figure(figsize=(12, 8))
plt.imshow(result_image)
plt.axis("off")
plt.show()